In [1]:
import sys
import numpy as np

# sys.path.append("../../../src/")
from Rain.Rain import Rain
# sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-05 22:58:49.771541: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-05 22:58:56.033914: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import os

def clean():
    folder_paths = ["logs", "../../../data/coord/", "../../../data/divider/", "../../../data/worker/"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [3]:
config = {
  "mode": {
      "type": "local",
      "params": {
        "num_of_workers": 3,
        "ips": ['172.190.116.144', '172.190.116.144', '172.190.116.144','127.0.0.1', '127.0.0.1', '127.0.0.1'], #[,'127.0.0.1', '127.0.0.1', '127.0.0.1'], 
        "ports": [50151, 50152, 50153, 50154, 50155, 50156]
        
      }
    },
  "temp_data_path": "../../../",
  "partitions": 3,
  "iterations": 1,
  "chunk_size": 1024*1024,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 1,
    "batch_size": 128,
  }
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )



In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()


In [7]:
model = create_model()
rain = Rain(config, model)

2023-07-05 22:59:10,193 [ERROR] [Rain] Error in the config: Error in partitions: argument of type 'int' is not iterable
2023-07-05 22:59:10,195 [DEBUG] [Rain] Rain is initialized
2023-07-05 22:59:10,196 [DEBUG] [Provisioner] Creating coordinator
2023-07-05 22:59:10,198 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/coord/
2023-07-05 22:59:10,199 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-05 22:59:10,201 [DEBUG] [LocalProvisioner] Provisioner is initialized
2023-07-05 22:59:10,203 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-05 22:59:10,204 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-05 22:59:10,206 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/


In [8]:
model = rain.train(X_train, y_train, strategy='async')

2023-07-05 22:59:10,341 [INFO] [Provisioner] provisioner is serving
2023-07-05 22:59:10,342 [DEBUG] [Provisioner] Starting coordinator
2023-07-05 22:59:10,346 [INFO] [Coordinator] coordinator is serving
2023-07-05 22:59:10,348 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-05 22:59:10,436 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-05 22:59:10,442 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-05 22:59:10,445 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-05 22:59:10,450 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker/
2023-07-05 22:59:10,459 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-05 22:59:10,462 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker/
2023-07-05 22:59:10,470 [INFO] [Worker_50152] Worker is running on port: 50152
2023-07-05 22:59:1

157/157 [==============================] - 11s 22ms/step - loss: 0.6995 - accuracy: 0.7790


2023-07-05 22:59:41,554 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}


sending data to divider


2023-07-05 22:59:41,561 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/3_3_trained.pkl from worker3
2023-07-05 22:59:41,770 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
2023-07-05 22:59:41,804 [DEBUG] [DeepLearning] Asynchronous update is done by worker 3
2023-07-05 22:59:41,842 [DEBUG] [DeepLearning] Iteration 1/1 complete for worker 3.
2023-07-05 22:59:46,035 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
2023-07-05 22:59:46,040 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_2_trained.pkl from worker2
2023-07-05 22:59:46,150 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
2023-07-05 22:59:46,153 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1


sending data to divider
sending data to divider


2023-07-05 22:59:46,523 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/2_2_trained.pkl from worker2 successfully
2023-07-05 22:59:46,540 [DEBUG] [DeepLearning] Asynchronous update is done by worker 2
2023-07-05 22:59:46,581 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
2023-07-05 22:59:46,592 [DEBUG] [DeepLearning] Iteration 1/1 complete for worker 2.
2023-07-05 22:59:46,626 [DEBUG] [DeepLearning] Asynchronous update is done by worker 1
2023-07-05 22:59:46,699 [DEBUG] [DeepLearning] Iteration 1/1 complete for worker 1.
2023-07-05 22:59:46,733 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
2023-07-05 22:59:46,734 [DEBUG] [Divider] Divider stopped serving
2023-07-05 22:59:46,736 [INFO] [Worker_50151] Worker stopped serving on port: 50151
2023-07-05 22:59:46,738 [INFO] [Worker_50152] Worker stopped serving on port: 50152
2023-07-05 22:59:46,742 [INFO] [Worker_50153] Worker stopped serving on p

In [9]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 1s 4ms/step - loss: 0.2494 - accuracy: 0.9266

Test accuracy: 92.7%


In [10]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-05 22:59:49,041 [INFO] [Provisioner] provisioner is serving
2023-07-05 22:59:49,044 [DEBUG] [Provisioner] Starting coordinator
2023-07-05 22:59:49,046 [INFO] [Coordinator] coordinator is serving
2023-07-05 22:59:49,080 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-05 22:59:49,083 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-05 22:59:49,084 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-05 22:59:49,086 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-05 22:59:49,088 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker/
2023-07-05 22:59:49,105 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-05 22:59:49,105 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-05 22:59:49,108 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker/
2023-07-05 22:59:4

157/157 [==============================] - 8s 21ms/step - loss: 0.3148 - accuracy: 0.9061


2023-07-05 23:00:03,869 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker{worker_id}
2023-07-05 23:00:03,877 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/3_1_trained.pkl from worker3
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/3_1_trained.pkl from worker3


sending data to divider
sending data to divider
157/157 [==============================] - ETA: 0s - loss: 0.3071 - accuracy: 0.9100

2023-07-05 23:00:03,896 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker{worker_id}
2023-07-05 23:00:03,900 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_1_trained.pkl from worker2
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/2_1_trained.pkl from worker2


157/157 [==============================] - 8s 22ms/step - loss: 0.3071 - accuracy: 0.9100


2023-07-05 23:00:03,941 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker{worker_id}
2023-07-05 23:00:03,945 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1
DEBUG:DividerAmbassador:divider begins downloading ../../..//RainData/divider/1_1_trained.pkl from worker1


sending data to divider


2023-07-05 23:00:04,478 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/3_1_trained.pkl from worker3 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/3_1_trained.pkl from worker3 successfully
2023-07-05 23:00:04,481 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/2_1_trained.pkl from worker2 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/2_1_trained.pkl from worker2 successfully
2023-07-05 23:00:04,492 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
DEBUG:DividerAmbassador:Downloaded ../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
2023-07-05 23:00:04,972 [DEBUG] [DeepLearning] Iteration 1/1 complete.
DEBUG:DeepLearning:Iteration 1/1 complete.
2023-07-05 23:00:05,111 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
DEBUG:DividerAmbassador:divider ambassador stopped serving
2023-07-05 23:00:05,117 [DEBUG] 

In [11]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 3ms/step - loss: 0.1647 - accuracy: 0.9494

Test accuracy: 94.9%
